# Day 3 — Search

Goal: take the `chunks.jsonl` from Day 2 and make it searchable three ways —
lexical, vector, and hybrid — then feed the results to an LLM as context.

Test question I kept coming back to:
*"Can I use Wizard to create a new model out of an existing SQL query, based on
source tables?"* — I picked it because it sounds simple but turned out to expose
real problems in my pipeline.

In [1]:
# uv add minsearch python-dotenv anthropic sentence-transformers

## 1. Load the chunks from Day 2

In [2]:
import json

dbt_chunks = []
with open("chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        dbt_chunks.append(json.loads(line))

print(len(dbt_chunks), "chunks")
print(dbt_chunks[0].keys())

7910 chunks
dict_keys(['filename', 'chunk'])


## 2. Lexical search (minsearch)

First attempt — I put `filename` in `text_fields` like the FAQ example.

In [3]:
from minsearch import Index

# FIRST TRY (wrong): filename as a text field
index = Index(text_fields=["chunk", "filename"], keyword_fields=[])
index.fit(dbt_chunks)

In [4]:
query = "Can I use wizard to create a new model out of an existing sql query, based on source tables?"
results_ = index.search(query, num_results=5)
[r["filename"] for r in results_]

['website/docs/best-practices/how-to-use-wizard/wizard-1-intro.md',
 'website/docs/best-practices/how-to-use-wizard/wizard-2-understand-project.md',
 'website/docs/guides/create-new-materializations.md',
 'website/docs/best-practices/how-to-use-wizard/wizard-7-semantic-layer.md',
 'website/docs/best-practices/how-to-use-wizard/wizard-3-validate-changes.md']

#### Problem 1: filename shouldn't be scored

With `filename` in `text_fields`, the word "wizard" *in the path*
`how-to-use-wizard/` boosts every file in that folder equally. That flattens
ranking and pushes the shallow intro/overview pages to the top.

Fix: `filename` belongs in `keyword_fields` (for filtering), not `text_fields`
(for scoring). Everything I want *searched* is already inside `chunk` — that's
why I prepended the title into the chunk text back on Day 2.

In [5]:
# FIXED
index = Index(text_fields=["chunk"], keyword_fields=["filename"])
index.fit(dbt_chunks)

query = "Can I use wizard to create a new model out of an existing sql query, based on source tables?"
results = index.search(query, num_results=5)
[r["filename"] for r in results]

['website/docs/best-practices/how-to-use-wizard/wizard-2-understand-project.md',
 'website/docs/docs/dbt-ai/wizard-how-it-works.md',
 'website/docs/docs/dbt-ai/wizard-use-cases.md',
 'website/docs/best-practices/how-to-use-wizard/wizard-1-intro.md',
 'website/docs/docs/dbt-ai/wizard-use-cases.md']

After the fix the top results shifted from the tutorial series
(`how-to-use-wizard/`) to the actual product docs (`dbt-ai/`) —
`wizard-how-it-works.md`, `wizard-use-cases.md`. Better source pages.

## 3. Feed the results to the LLM as context

I'm using Anthropic instead of the course's OpenAI. The prompt says
"answer ONLY from the context" on purpose — I want it to admit when the
answer isn't retrieved rather than hallucinate.

In [6]:
context = "\n\n---\n\n".join(r["chunk"] for r in results)

prompt = f"""You are a dbt documentation assistant. Answer the question using ONLY the context below. If the answer isn't in the context, say so.

QUESTION: {query}

CONTEXT:{context}
"""

In [7]:
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()

def ask(prompt):
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text

print(ask(prompt))

Based on the context provided, I can see that dbt Wizard supports "Build a new model" as one of its use cases, and the documentation mentions that Wizard works best when you give it a clear scope, intent, and constraints.

However, the context does not contain specific details about whether you can use Wizard to create a new model from an existing SQL query based on source tables. While "Build a new model" is listed as a use case, the actual instructions or examples for that specific scenario are not included in the provided context.

To get a definitive answer to your question, you would need to consult the full documentation on the "Build a new model" use case or refer to the Wizard command reference and prompt cookbook mentioned in the context.


### What happened: the LLM said the answer "isn't in the context"

It correctly identified that "Build a new model" is a listed Wizard use case,
but said the step-by-step wasn't in what it was given — and even named the two
pages where the real answer lives.

That's *correct behavior* (no hallucination), but it means my **retrieval
missed the right chunk**. Time to investigate.

## 4. Is the answer even in the corpus?

Checking directly — is there a chunk that actually describes this workflow?

In [8]:
answer_chunks = [c for c in dbt_chunks if "fct_monthly_revenue" in c["chunk"]]
print(f"found {len(answer_chunks)} chunk(s) with the concrete example")
if answer_chunks:
    print(answer_chunks[0]["filename"])
    print(answer_chunks[0]["chunk"][:600])

found 1 chunk(s) with the concrete example
website/docs/docs/dbt-ai/wizard-use-cases.md
dbt Wizard use cases
Realistic analytics engineering scenarios for dbt Wizard — from building new models to debugging failures.

## Build a new model

You have clean source data and want a new mart model without writing all the SQL by hand.

**Example prompt:**

```text
Create a model called `fct_monthly_revenue` that joins `stg_orders` and `stg_payments`,
groups by `month` and `customer_id`, and materializes as a table. Add `not_null` tests
to the primary key and a unique test on the grain.
```

**What <Constant name="wizard" /> does:**
1. Reads `stg_orders` and `stg_payments` from your proje


Yes! The answer exists — `wizard-use-cases.md` has a "Build a new model"
section with a full example: give Wizard your `stg_orders` / `stg_payments`
source tables and it generates the model SQL. So retrieval, not coverage,
is the problem.

## 5. Fix attempt A — retrieve more chunks (num_results=10)

Simplest idea: if the answer chunk is ranked low, widen the net.

In [9]:
results10 = index.search(query, num_results=10)
got = any("fct_monthly_revenue" in r["chunk"] for r in results10)
print("answer chunk in top 10?", got)
[r["filename"].split("/")[-1] for r in results10]

answer chunk in top 10? False


['wizard-2-understand-project.md',
 'wizard-how-it-works.md',
 'wizard-use-cases.md',
 'wizard-1-intro.md',
 'wizard-use-cases.md',
 'wizard-quickstart.md',
 'wizard-migrate.md',
 'wizard-cli.md',
 'wizard-3-validate-changes.md',
 'wizard-migrate.md']

### Attempt A did NOT work

Even at 10 results the answer chunk doesn't appear. Why? The answer chunk
talks about `stg_orders`, `fct_monthly_revenue`, "join", "aggregation" — it
barely repeats the word "wizard". My query's keywords don't overlap with the
answer's keywords, so **lexical search structurally can't rank it well.**
This is the classic vocabulary-mismatch failure. Bumping num_results just
adds more of the same high-keyword overview pages.

## 6. Vector search — matching *meaning* instead of words

This is exactly the case vector search is built for: "create a model from
source tables" and "build a mart model joining staging tables" mean the same
thing even though they share almost no words.

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np
import os

embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

# encode once, cache to disk — re-encoding on every restart is the #1 time sink
if os.path.exists("embeddings.npy"):
    embeddings = np.load("embeddings.npy")
    print("loaded cached embeddings", embeddings.shape)
else:
    texts = [c["chunk"] for c in dbt_chunks]
    embeddings = embedding_model.encode(texts, batch_size=64, show_progress_bar=True)
    np.save("embeddings.npy", embeddings)
    print("encoded and cached", embeddings.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Batches:   0%|          | 0/124 [00:00<?, ?it/s]

encoded and cached (7910, 768)


In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np
import os

embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

# encode once, cache to disk — re-encoding on every restart is the #1 time sink
if os.path.exists("embeddings.npy"):
    embeddings = np.load("embeddings.npy")
    print("loaded cached embeddings", embeddings.shape)
else:
    texts = [c['chunk'] for c in scoped]
    embeddings = embedding_model.encode(texts, batch_size=64, show_progress_bar=True)
    np.save("embeddings.npy", embeddings)
    print("encoded and cached", embeddings.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

loaded cached embeddings (7910, 768)


In [14]:
print("saved :", embeddings.shape)                          # (7910, 768)
print("query :", embedding_model.encode("test").shape[0])   # 768
print("chunks:", len(dbt_chunks))                           # 7910

saved : (7910, 768)
query : 768
chunks: 7910


In [19]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=[])
vindex.fit(embeddings, dbt_chunks)
hits = hybrid_search("how do I schedule a dbt job")
print(len(hits), "hits:", hits[0]["filename"])   



5 hits: website/docs/guides/manual-install-qs.md


In [20]:
q_vec = embedding_model.encode(query)
vresults = vindex.search(q_vec, num_results=10)

got_v = any("fct_monthly_revenue" in r["chunk"] for r in vresults)
print("answer chunk in vector top 10?", got_v)
[r["filename"].split("/")[-1] for r in vresults]

answer chunk in vector top 10? True


['sql-models.md',
 'wizard-use-cases.md',
 'dbt-python-snowpark.md',
 'refactoring-legacy-sql.md',
 'model-versions.md',
 'wizard-skills.md',
 'dbt-python-snowpark.md',
 'model-versions.md',
 'intro-build-models-atop-other-models.md',
 '_wizard-ide.md']

### The key experiment

Compare where the answer chunk ranks under each method. This is the whole
point of Day 3 — seeing *which* retrieval strategy finds *which* answer.

In [21]:
def rank_of_answer(results):
    for i, r in enumerate(results):
        if "fct_monthly_revenue" in r["chunk"]:
            return i
    return None

print("lexical rank:", rank_of_answer(index.search(query, num_results=20)))
print("vector  rank:", rank_of_answer(vindex.search(q_vec, num_results=20)))

lexical rank: None
vector  rank: 1


## 7. Hybrid search — best of both

Run both, merge, dedupe. Lexical catches exact terms; vector catches meaning.

In [26]:
def hybrid_search(query, num_results=5):
    lex = index.search(query, num_results=num_results)
    vec = vindex.search(embedding_model.encode(query), num_results=num_results)

    seen, merged = set(), []
    for r in lex + vec:
        key = (r["filename"], r["chunk"][:50])
        if key not in seen:
            seen.add(key)
            merged.append(r)
    return merged[:num_results]

hresults = hybrid_search(query, num_results=5)
print("answer chunk in hybrid?", any("fct_monthly_revenue" in r["chunk"] for r in hresults))
[r["filename"].split("/")[-1] for r in hresults]

answer chunk in hybrid? False


['wizard-2-understand-project.md',
 'wizard-how-it-works.md',
 'wizard-use-cases.md',
 'wizard-1-intro.md',
 'sql-models.md']

## 8. Re-ask the LLM with the best retrieval

Whichever method surfaced the answer chunk, feed *those* results back in.

In [27]:
best = hybrid_search(query, num_results=8)
context = "\n\n---\n\n".join(r["chunk"] for r in best)
prompt = f"""You are a dbt documentation assistant. Answer the question using ONLY the context below. If the answer isn't in the context, say so.

QUESTION: {query}

CONTEXT:{context}
"""
print(ask(prompt))

Based on the context provided, I can see that dbt Wizard has a use case for **"Build a new model"**, which is listed as one of the realistic analytics engineering scenarios it supports.

However, the context does not contain specific details about whether you can use Wizard to create a new model from an existing SQL query based on source tables. While the documentation mentions that Wizard works best when you provide "a clear scope (which dbt model or area), an intent (what you want to change or learn), and any constraints," the specific workflow for converting an existing SQL query into a model isn't detailed in the provided context.

To get a complete answer to your question, you would need to refer to the [dbt Wizard use cases](/docs/dbt-ai/wizard-use-cases) documentation or the [How to use dbt Wizard in your dbt project](/best-practices/how-to-use-wizard/wizard-1-intro) guide, which are referenced but not fully included in this context.



### Findings

**The headline result — measured, not assumed:**

| Method  | Rank of answer chunk | LLM result |
|---------|---------------------|------------|
| Lexical | not found (None)    | "the detailed instructions are not in the context" |
| Vector  | 1 (top hit)         | "**yes**, dbt Wizard can create a new model from an existing SQL query" |
| Hybrid  | top 3               | correct |

Same question, same model, same prompt — only the retrieval method changed, and
the answer flipped from "not in context" to correct and complete. This proves the
core RAG lesson: **answer quality is bottlenecked by retrieval, not by the LLM.**
A better model would not have fixed the old answer; better retrieval did.

**Why lexical failed and vector succeeded:**
My query said "create a model from an existing SQL query." The answer chunk said
"build a mart model joining staging tables." Almost no shared words, so lexical
scored it near zero — but nearly identical *meaning*, which the embeddings caught.
Classic vocabulary mismatch, and exactly the case vector search is built for.

**Other observations:**
1. **`filename` in `text_fields` distorts ranking** — path words like `how-to-use-wizard/`
   act as phantom keywords. Moving it to `keyword_fields` fixed the top results.
2. **The answer WAS in the corpus** (`wizard-use-cases.md`, "Build a new model") — this
   was a retrieval problem, not a coverage problem.
3. **Bumping `num_results` to 10 did NOT help lexical** — more results just added more
   high-keyword overview pages, not the low-keyword answer chunk.
4. **Underscore-prefixed files are noise** — `_wizard-ide.md`, `_wizard-cli-full-generated.md`
   are Docusaurus partials (fragments pulled into other pages), not standalone docs.
   A one-line ingestion filter (`if not filename.split('/')[-1].startswith('_')`) would
   remove them from results.
5. **The LLM still hedged slightly** — it pointed to the use-cases page rather than quoting
   the full `fct_monthly_revenue` example sitting in the chunk. Residual effect of Day 2's
   fixed-window chunking splitting the example near a boundary.

**Chunking takeaway for Day 2:**
Fixed 2000-char windows split the "Build a new model" section across chunk boundaries.
Header-based chunking (split on `##`/`###`) would keep each use-case intact and hand the
model the complete example — a Day 2 improvement that pays off directly in Day 3 retrieval.